# Notebook 01 — SFT Dataset A (Crownelius / Opus Reasoning)
**Assignment 04 | Track 2, Option D | NLP with Deep Learning — IBA**

**Model:** `Qwen/Qwen3-0.6B`  
**Dataset:** `Crownelius/Opus-4.6-Reasoning-3300x` — 2,160 rows with `problem`, `thinking`, `solution`, `difficulty`, `category`

**Ablation design (5 trials):**
| Trial | Data % | LoRA Rank | Target Modules | LR | Batch | Grad Accum | Epochs | Purpose |
|-------|--------|-----------|----------------|----|-------|------------|--------|---------|
| T1 | 25% | 16 | q,k,v,o | 2e-4 | 2 | 4 | 1 | Data efficiency — low |
| T2 | 50% | 16 | q,k,v,o | 2e-4 | 2 | 4 | 1 | Data efficiency — mid |
| T3 | 100% | 16 | q,k,v,o | 2e-4 | 2 | 4 | 1 | Data efficiency — full |
| T4 | 100% | 32 | q,k,v,o,gate,up,down | 1e-4 | 4 | 4 | 2 | LoRA capacity — wider |
| T5 | 100% | 64 | q,k,v,o | 5e-5 | 2 | 8 | 2 | LoRA capacity — deeper |

**Run this on:** Any CUDA GPU machine (local 3090, Kaggle T4, Colab)  
**Prerequisite:** Run `00_baseline.ipynb` first to generate `prompts.json` and `gold_answers.json`

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Unsloth: auto-detects CUDA version and installs the correct build
!pip install -q "unsloth[cu124-torch260] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || \
 pip install -q unsloth
!pip install -q trl peft transformers datasets accelerate bitsandbytes huggingface_hub python-dotenv

In [ ]:
import os, json, time, re, torch, pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import InferenceClient
from dotenv import load_dotenv

load_dotenv()

# ── CONFIG ────────────────────────────────────────────────────────────────────
HF_TOKEN    = os.getenv("HF_TOKEN")   # loaded from .env
BASE_MODEL  = "Qwen/Qwen3-0.6B"
DATASET_ID  = "Crownelius/Opus-4.6-Reasoning-3300x"
JUDGE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LEN = 2048
OUTPUT_DIR  = "./adapters_datasetA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## Step 1 — Load Dataset and Verify Columns

In [ ]:
ds = load_dataset(DATASET_ID, split="train")
print(f"Dataset size: {len(ds)} rows")
print(f"Columns: {ds.features}")
print("\nSample row:")
sample = ds[0]
for k, v in sample.items():
    print(f"  {k}: {str(v)[:120]}")

## Step 2 — Format Dataset with Chat Template

We include the `thinking` field in the assistant turn using `<think>` tags.
This is the key structural advantage of Dataset A — explicit chain-of-thought supervision.

In [ ]:
def get_tokenizer():
    """Load tokenizer once to format examples."""
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    return tok

tokenizer_for_format = get_tokenizer()

def format_row(row):
    """Format a Crownelius row into a chat-template string.
    Includes thinking trace for CoT supervision.
    """
    thinking = row.get("thinking", "").strip()
    solution = row.get("solution", "").strip()

    if thinking:
        assistant_content = f"<think>{thinking}</think>\n{solution}"
    else:
        assistant_content = solution

    messages = [
        {"role": "system",    "content": "You are a helpful reasoning assistant. Think step by step."},
        {"role": "user",      "content": row["problem"].strip()},
        {"role": "assistant", "content": assistant_content},
    ]
    return tokenizer_for_format.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

ds_formatted = ds.map(lambda row: {"text": format_row(row)})
print("Sample formatted text (first 600 chars):")
print(ds_formatted[0]["text"][:600])

## Step 3 — Load Prompts and Gold Answers for Evaluation

In [ ]:
# Load from notebook 00 outputs — same working directory on Kaggle
with open("prompts.json") as f:
    PROMPTS = json.load(f)
with open("gold_answers.json") as f:
    gold_answers = json.load(f)

print(f"Loaded {len(PROMPTS)} prompts and {len(gold_answers)} gold answers.")

JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""


def _parse_judge_output(out):
    score_match = re.search(r"SCORE\s*:\s*([1-5])", out, re.IGNORECASE)
    reason_match = re.search(r"REASON\s*:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    if not score_match:
        raise ValueError(f"Judge response did not include SCORE 1-5. Raw output: {out!r}")
    reason = reason_match.group(1).strip() if reason_match else out.strip()
    return int(score_match.group(1)), reason


def judge_response(question, gold, response):
    client = InferenceClient(JUDGE_MODEL, token=HF_TOKEN)
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, gold=gold, response=response)
    mistral_prompt = f"<s>[INST] {prompt} [/INST]"
    try:
        out = client.text_generation(
            mistral_prompt,
            max_new_tokens=120,
            do_sample=False,
            return_full_text=False,
        ).strip()
        return _parse_judge_output(out)
    except Exception as text_error:
        try:
            out = client.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=120,
                temperature=0.0,
            ).choices[0].message.content.strip()
            return _parse_judge_output(out)
        except Exception as chat_error:
            return 0, f"ERROR: text_generation failed: {text_error}; chat_completion failed: {chat_error}"



def _apply_chat_template(tokenizer, messages):
    """Handles BatchEncoding (transformers 4.50+) and plain tensor (older)."""
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    return encoded["input_ids"] if not isinstance(encoded, torch.Tensor) else encoded


def eval_model_on_prompts(model, tokenizer, trial_name):
    """Run fine-tuned model on all 10 prompts and get judge scores."""
    FastLanguageModel.for_inference(model)
    records = []
    for p in PROMPTS:
        messages = [
            {"role": "system",  "content": "You are a helpful reasoning assistant. Think step by step."},
            {"role": "user",    "content": p["prompt"]}
        ]
        input_ids = _apply_chat_template(tokenizer, messages).to(model.device)
        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=512, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        score, reason = judge_response(p["prompt"], gold_answers[p["id"]], response)
        records.append({
            "trial": trial_name, "prompt_id": p["id"], "type": p["type"],
            "response": response, "score": score, "reason": reason
        })
        print(f"  [{trial_name}] {p['id']}: score={score}")
        time.sleep(0.5)
    return records

## Step 4 — Trial Loop (5 Trials)

Each trial: train → save adapter → evaluate on 10 prompts → judge score

In [ ]:
TRIAL_CONFIGS = [
    {
        "name": "T1", "data_pct": 0.25, "rank": 16,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
        "purpose": "Data efficiency — 25%"
    },
    {
        "name": "T2", "data_pct": 0.50, "rank": 16,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
        "purpose": "Data efficiency — 50%"
    },
    {
        "name": "T3", "data_pct": 1.00, "rank": 16,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
        "purpose": "Data efficiency — 100%"
    },
    {
        "name": "T4", "data_pct": 1.00, "rank": 32,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        "lr": 1e-4, "batch": 4, "grad_accum": 4, "epochs": 2,
        "purpose": "LoRA capacity — rank 32, wider modules"
    },
    {
        "name": "T5", "data_pct": 1.00, "rank": 64,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 5e-5, "batch": 2, "grad_accum": 8, "epochs": 2,
        "purpose": "LoRA capacity — rank 64, deeper"
    },
]

# fp16/bf16 — only set when actually on GPU; CPU training uses full fp32
use_fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"fp16={use_fp16} | bf16={use_bf16}")

all_records = []
trial_summary = []

for cfg in TRIAL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Trial {cfg['name']}: {cfg['purpose']}")
    print(f"  data={cfg['data_pct']*100:.0f}%, rank={cfg['rank']}, lr={cfg['lr']}, "
          f"batch={cfg['batch']}, grad_accum={cfg['grad_accum']}, epochs={cfg['epochs']}")
    print('='*60)

    n = int(len(ds_formatted) * cfg["data_pct"])
    ds_train = ds_formatted.select(range(n))
    split = ds_train.train_test_split(test_size=0.1, seed=42)
    train_ds = split["train"]
    eval_ds  = split["test"]
    print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True, token=HF_TOKEN
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=cfg["rank"],
        target_modules=cfg["target_modules"],
        lora_alpha=cfg["rank"],
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing=True,
    )

    adapter_path = os.path.join(OUTPUT_DIR, cfg["name"])

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=SFTConfig(
            output_dir=adapter_path,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            per_device_train_batch_size=cfg["batch"],
            gradient_accumulation_steps=cfg["grad_accum"],
            learning_rate=cfg["lr"],
            num_train_epochs=cfg["epochs"],
            eval_strategy="epoch",
            save_strategy="epoch",
            # load_best_model_at_end NOT used — incompatible with Unsloth 4-bit quantized models
            fp16=use_fp16,
            bf16=use_bf16,
            logging_steps=10,
            report_to="none",
        ),
    )

    train_result = trainer.train()
    val_loss = trainer.evaluate()["eval_loss"]

    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"  Adapter saved to: {adapter_path}")
    print(f"  Train loss: {train_result.training_loss:.4f} | Val loss: {val_loss:.4f}")

    print("  Evaluating on 10 baseline prompts...")
    records = eval_model_on_prompts(model, tokenizer, cfg["name"])
    for r in records:
        r.update({"model": BASE_MODEL, "stage": "sft_datasetA",
                  "data_pct": cfg["data_pct"], "rank": cfg["rank"],
                  "lr": cfg["lr"], "val_loss": val_loss})
    all_records.extend(records)

    avg_score = sum(r["score"] for r in records) / len(records)
    trial_summary.append({
        "trial": cfg["name"], "purpose": cfg["purpose"],
        "data_pct": f"{cfg['data_pct']*100:.0f}%",
        "rank": cfg["rank"], "lr": cfg["lr"],
        "batch": cfg["batch"], "epochs": cfg["epochs"],
        "val_loss": round(val_loss, 4),
        "avg_judge_score": round(avg_score, 2),
        "adapter_path": adapter_path,
    })
    print(f"  Avg judge score: {avg_score:.2f}")

    del model, trainer
    torch.cuda.empty_cache()

print("\nAll 5 trials complete.")

## Step 5 — Select Best Trial and Save Results

In [ ]:
df_summary = pd.DataFrame(trial_summary)
print("\n── Trial Results ──")
print(df_summary[["trial","purpose","data_pct","rank","lr","epochs","val_loss","avg_judge_score"]].to_string(index=False))

# Select best: highest avg_judge_score, tie-break on lowest val_loss
best = df_summary.sort_values(
    ["avg_judge_score", "val_loss"], ascending=[False, True]
).iloc[0]

print(f"\n── Best Trial: {best['trial']} ──")
print(f"  Purpose:         {best['purpose']}")
print(f"  Avg judge score: {best['avg_judge_score']}")
print(f"  Val loss:        {best['val_loss']}")
print(f"  Adapter saved:   {best['adapter_path']}")

# Save
df_summary.to_csv("trial_results_datasetA.csv", index=False)
pd.DataFrame(all_records).to_csv("all_scores_datasetA.csv", index=False)
with open("best_trial_datasetA.json", "w") as f:
    json.dump(best.to_dict(), f, indent=2)

print("\nSaved: trial_results_datasetA.csv, all_scores_datasetA.csv, best_trial_datasetA.json")